In [1]:
%load_ext dotenv
%dotenv

import os

%cd {os.getenv("PROJECT_ROOT") or ".."}

%load_ext autoreload
%autoreload 1

from IPython.display import display

cannot find .env file
/home/aris/projects/evagpt


In [2]:
import logging

import pandas as pd

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

In [3]:
from pandarallel import pandarallel

pandarallel.initialize(nb_workers=os.cpu_count(), progress_bar=True, verbose=0)

In [4]:
def show_df(df: pd.DataFrame) -> None:
    display(df.head())
    print(df.shape)

In [12]:
import torch
from torch import nn

C = 96

c_attn = nn.Linear(C, C)

x = torch.randn(8, 20, C)

In [29]:
Q, K, V = c_attn(x).chunk(3, dim=-1)

Q.shape

torch.Size([8, 20, 32])

In [16]:
Q.is_contiguous()

False

In [17]:
Q.view(8, 20, 4, 8).shape

torch.Size([8, 20, 4, 8])

In [20]:
Q.view(8, 20, 4, 8).transpose(-1, -2).is_contiguous()

False

In [23]:
Q.view(8, 20, 4, 8).transpose(1, 2).stride(-1)

1

In [26]:
Q.view(8, 20, 4, 8).transpose(1, 2).stride(-2)

96

In [27]:
Q.view(8, 20, 4, 8).transpose(1, 2).transpose(-1, -2).stride(-2)

1

In [30]:
Q = Q.view(8, 20, 4, 8).transpose(1, 2)
K = K.view(8, 20, 4, 8).transpose(1, 2)

In [31]:
qkt = torch.einsum("...ij,...kj->...ik", Q, K)
qkt.shape

torch.Size([8, 4, 20, 20])

In [32]:
qkt.is_contiguous()

True